In [2]:
# simple imports for dashboard
from jupyter_dash import JupyterDash

# dashboard pieces
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px

# data stuff
import pandas as pd
import base64

# helps Codio proxy
JupyterDash.infer_jupyter_proxy_config()

# import our CRUD code
from CRUD_Python_Module import AnimalShelter

# connect to database
db = AnimalShelter()

# get all data from database
df = pd.DataFrame.from_records(db.read({}))

# remove id column
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)


# load Grazioso logo
# NOTE: logo must be in same folder as your ipynb file
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())


# start our app
app = JupyterDash(__name__)


# layout = what the page looks like
app.layout = html.Div([

    # show logo and title
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
             style={'height': '100px'}),
    html.H1("Grazioso Salvare Dashboard - Cecily Boucher"),

    # radio buttons for selecting filter
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'Reset', 'value': 'reset'},
            {'label': 'Water Rescue', 'value': 'water'},
            {'label': 'Mountain/Wilderness', 'value': 'mountain'},
            {'label': 'Disaster/Tracking', 'value': 'disaster'}
        ],
        value='reset',
        inline=True
    ),

    # main data table
    dash_table.DataTable(
        id='datatable-id',
        data=df.to_dict('records'),
        columns=[{"name": i, "id": i} for i in df.columns],
        page_size=10
    ),

    html.Br(),

    # graph + map side by side
    html.Div(style={'display': 'flex'}, children=[
        html.Div(id='graph-id', style={'width': '50%'}),
        html.Div(id='map-id', style={'width': '50%'})
    ])
])


# filter and update table

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    # water rescue filters
    if filter_type == 'water':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    # mountain rescue
    elif filter_type == 'mountain':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler"
            ]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    # disaster
    elif filter_type == 'disaster':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler"
            ]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    # reset = all animals
    else:
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


# pie chart

@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graph(viewData):

    # if nothing yet, show full df
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

    # make a pie showing breed
    fig = px.pie(dff, names='breed', title='Breed Distribution')

    return [dcc.Graph(figure=fig)]


# Map
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    
    if not viewData:
        return html.Div("No data to show")
    
    dff = pd.DataFrame(viewData)
    
    # if no selection, choose first row
    row = 0 if not index else index[0]
    
    # use fallback for missing map values
    lat = dff.iloc[row].get("location_lat", 30.75)
    lon = dff.iloc[row].get("location_long", -97.48)
    breed = dff.iloc[row].get("breed", "Unknown Breed")
    name = dff.iloc[row].get("name", "Unknown Animal")
    
    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[lat, lon],
            zoom=12,
            children=[
                dl.TileLayer(),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(breed),
                        dl.Popup([
                            html.H3("Animal Name"),
                            html.P(name)
                        ])
                    ]
                )
            ]
        )
    ]



# run app
app.run_server()


Dash app running on https://germanyaztec-prestocola-3000.codio.io/proxy/8050/
